# 🗂️ 공공데이터 API 실습 모음집
**대상:** 빅데이터 과정 수강생  
**목표:** 다양한 주제의 공공 API를 직접 찾고, 호출하고, 데이터를 분석한다

---
## 📌 공통 패턴 (모든 API에 동일하게 적용!)
```python
import requests

API_KEY = "본인키".strip()          # 1. 키 설정
url = "https://apis.data.go.kr/..." # 2. 주소 설정
params = {"serviceKey": API_KEY}    # 3. 파라미터 설정
response = requests.get(url, params=params)  # 4. 요청
data = response.json()              # 5. JSON 변환
```

---
## 📋 실습 목록
| 번호 | 주제 | API | 포털 검색어 |
|------|------|-----|------------|
| 1 | 🌤️ 날씨 | 기상청 단기예보 | `기상청 단기예보` |
| 2 | 🍽️ 음식/위생 | 식품안전나라 음식점 | `일반음식점` |
| 3 | 🏥 의료 | 전국 병원/약국 | `건강보험심사평가원 병원` |
| 4 | 🎭 문화/관광 | 한국관광공사 행사정보 | `한국관광공사 관광정보` |
| 5 | 🚗 교통 | 국가교통정보 주차장 | `주차장 정보` |
| 6 | 💊 의약품 | 의약품 정보 | `의약품개요정보` |

> 💡 **팁:** data.go.kr 검색창에 위 검색어를 입력하고 직접 찾아보세요!

In [54]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv('OPENAPI_API_KEY')

In [66]:
# ✅ 공통 설정 - 모든 실습에서 사용
import requests
import json
import pandas as pd
from datetime import datetime

def pretty(data):
    """JSON 보기 좋게 출력"""
    print(json.dumps(data, ensure_ascii=False, indent=2))

def check_response(response):
    """응답 상태 확인"""
    if response.status_code == 200:
        print("✅ 성공! (200)")
    elif response.status_code == 401:
        print("🔴 API 키 오류 (401) - 키를 확인하세요")
    elif response.status_code == 400:
        print("🔴 파라미터 오류 (400) - 필수값을 확인하세요")
    else:
        print(f"🔴 오류: {response.status_code}")
    return response.status_code == 200

print("공통 설정 완료! ✅")

공통 설정 완료! ✅


---
# 🌤️ 실습 1. 기상청 단기예보

**포털 검색:** `기상청 단기예보 조회서비스`  
**포털 URL:** https://www.data.go.kr/data/15084084/openapi.do

### 📝 API 문서 탐색 미션
```
□ 초단기실황조회 엔드포인트: getUltraSrtNcst
□ 필수 파라미터: base_date, base_time, nx, ny
□ nx, ny 란? → 기상청 격자 좌표 (위경도 아님!)
□ 대구광역시 좌표: nx=89, ny=90
```

### 날씨 카테고리 코드표
| 코드 | 항목 | 단위 |
|------|------|------|
| T1H | 기온 | ℃ |
| RN1 | 1시간 강수량 | mm |
| REH | 습도 | % |
| WSD | 풍속 | m/s |
| PTY | 강수형태 | 코드 |

> 강수형태: 0=없음, 1=비, 2=비/눈, 3=눈, 5=빗방울, 6=빗방울/눈날림, 7=눈날림

In [67]:
# 🌤️ 기상청 초단기실황 - 현재 날씨
now = datetime.now()
base_date = now.strftime("%Y%m%d")
base_time = now.strftime("%H00")  # 정시 기준

url_weather = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

params_weather = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": "89",   # 대구광역시
    "ny": "90"
}

print(f"요청 날짜: {base_date}, 시간: {base_time}")
response_w = requests.get(url_weather, params=params_weather)

if check_response(response_w):
    data_w = response_w.json()
    items = data_w['response']['body']['items']['item']
    print(f"\nitem 리스트의 첫 번째 요소 : {items[0]}")

    print("\n📊 현재 날씨 데이터:")
    category_names = {
        "T1H": "기온(℃)", "RN1": "강수량(mm)",
        "REH": "습도(%)", "WSD": "풍속(m/s)", "PTY": "강수형태"
    }
    for item in items:
        name = category_names.get(item['category'], item['category'])
        print(f"    {name}: {item['obsrValue']}")

요청 날짜: 20260602, 시간: 1700
✅ 성공! (200)

item 리스트의 첫 번째 요소 : {'baseDate': '20260602', 'baseTime': '1700', 'category': 'PTY', 'nx': 89, 'ny': 90, 'obsrValue': '0'}

📊 현재 날씨 데이터:
    강수형태: 0
    습도(%): 77
    강수량(mm): 0
    기온(℃): 23.1
    UUU: -0.4
    VEC: 96
    VVV: 0
    풍속(m/s): 0.5


In [69]:
# 🎯 미션: 내가 원하는 도시 날씨 조회
# nx, ny 좌표표: https://www.weather.go.kr/w/observation/land/grid.do 에서 확인

# 주요 도시 좌표
cities = {
    "서울": (60, 127),
    "부산": (98, 76),
    "대구": (89, 90),
    "인천": (55, 124),
    "광주": (58, 74),
    "대전": (67, 100)
}

# 여기서 도시를 선택하세요!
my_city = "부산"  
nx, ny = cities[my_city]

print(f"선택한 도시: {my_city} (nx={nx}, ny={ny})")

params_weather = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": "98",   
    "ny": "76"
}

print(f"요청 날짜: {base_date}, 시간: {base_time}")
response = requests.get(url_weather, params=params_weather)

if check_response(response_w):
    data_w = response_w.json()
#     item = data_w['response']['body']['items']['item']

#     print("\n📊 현재 날씨 데이터:")
#     for item in items:
#         name = category_names.get(item['category'], item['category'])
#         print(f"    {name}: {item['obsrValue']}")
data_w

선택한 도시: 부산 (nx=98, ny=76)
요청 날짜: 20260602, 시간: 1700
✅ 성공! (200)


{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL_SERVICE'},
  'body': {'dataType': 'JSON',
   'items': {'item': [{'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'PTY',
      'nx': 89,
      'ny': 90,
      'obsrValue': '0'},
     {'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'REH',
      'nx': 89,
      'ny': 90,
      'obsrValue': '77'},
     {'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'RN1',
      'nx': 89,
      'ny': 90,
      'obsrValue': '0'},
     {'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'T1H',
      'nx': 89,
      'ny': 90,
      'obsrValue': '23.1'},
     {'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'UUU',
      'nx': 89,
      'ny': 90,
      'obsrValue': '-0.4'},
     {'baseDate': '20260602',
      'baseTime': '1700',
      'category': 'VEC',
      'nx': 89,
      'ny': 90,
      'obsrValue': '96'},
     {'baseDate': '20260602',
      'ba

---
# 🍽️ 실습 2. 전국 일반음식점 정보

**포털 검색:** `일반음식점`  
**포털 URL:** https://www.data.go.kr/data/15096283/standard.do

> 이 API는 각 지자체에서 제공하므로 **지역별로 별도 신청** 필요!  
> 대구시: `대구 일반음식점` 으로 검색

### 📝 탐색 미션
```
□ 어떤 필드들이 있나요? (업소명, 주소, 업종, ...)
□ 필수 파라미터는?
□ 페이징 파라미터는? (numOfRows, pageNo)
```

In [ ]:
# 🍽️ 일반음식점 정보 조회
# ※ 지역마다 URL이 다릅니다! Swagger에서 확인한 URL을 입력하세요

url_food = "https://apis.data.go.kr/6270000/rstrntService/getRstrntList"  # 대구 예시

params_food = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "type": "json"
}

response_f = requests.get(url_food, params=params_food)
print("응답 코드:", response_f.status_code)

if response_f.status_code == 200:
    data_f = response_f.json()
    pretty(data_f)

In [ ]:
# 🎯 미션: 음식점 데이터를 pandas DataFrame으로 만들기
# (데이터 구조를 먼저 확인한 후 필드명에 맞게 수정하세요)

# 예시 - 응답 구조에 맞게 수정 필요
# items = data_f["response"]["body"]["items"]["item"]
# df = pd.DataFrame(items)
# print(df.head())
# print("\n컬럼 목록:", df.columns.tolist())

print("↑ 주석을 해제하고 실행해보세요!")

---
# 🏥 실습 3. 전국 병원/약국 정보

**포털 검색:** `건강보험심사평가원 병원정보서비스`  
**포털 URL:** https://www.data.go.kr/data/15001698/openapi.do

### 📝 탐색 미션
```
□ 병원 조회 엔드포인트: getBsshInfoInqire
□ 약국 조회 엔드포인트: getParmacyBasisList
□ 지역코드: Q0 (서울), Q3 (부산), Q4 (대구), Q5 (인천)
```

In [ ]:
# 🏥 대구 지역 병원 정보 조회
url_hospital = "https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList"

params_hospital = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "sidoCd": "410000",  # 대구광역시 코드
    "_type": "json"
}

response_h = requests.get(url_hospital, params=params_hospital)
print("응답 코드:", response_h.status_code)

if response_h.status_code == 200:
    data_h = response_h.json()
    items = data_h["response"]["body"]["items"]["item"]
    
    print(f"\n총 {data_h['response']['body']['totalCount']}개 병원")
    print("\n병원 목록 (10개):")
    for h in items:
        print(f"  🏥 {h['yadmNm']} | {h['clCdNm']} | {h['addr']}")

In [ ]:
# 💊 약국 정보 조회
url_pharmacy = "https://apis.data.go.kr/B551182/pharmacyInfoService/getParmacyBasisList"

params_ph = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "sidoCd": "410000",  # 대구
    "_type": "json"
}

response_ph = requests.get(url_pharmacy, params=params_ph)

if check_response(response_ph):
    data_ph = response_ph.json()
    items_ph = data_ph["response"]["body"]["items"]["item"]
    
    print("\n약국 목록:")
    for p in items_ph:
        print(f"  💊 {p['yadmNm']} | {p['addr']}")

In [ ]:
# 🎯 미션: 내 주변 특정 진료과목 병원 찾기
# 진료과목 코드 예시: D001=내과, D002=신경과, D003=정신건강의학과
#                   D006=소아청소년과, D013=피부과, D014=비뇨의학과

# 여기에 코드를 작성해보세요!
target_dept = "D013"  # ← 원하는 진료과목 코드로 바꾸세요

# params에 dgsbjtCd 파라미터를 추가해보세요
# params_hospital["dgsbjtCd"] = target_dept


---
# 🎭 실습 4. 한국관광공사 문화/행사 정보

**포털 검색:** `한국관광공사 관광정보 서비스`  
**포털 URL:** https://www.data.go.kr/data/15101578/openapi.do

> ⚠️ 이 API는 **별도 사이트에서도 신청 가능!**  
> https://api.visitkorea.or.kr/ 에서 회원가입 후 키 발급

### 📝 탐색 미션
```
□ 행사정보 엔드포인트: searchFestival
□ 지역기반 관광정보: areaBasedList
□ 지역코드: 1=서울, 2=인천, 4=대구, 5=광주, 6=부산
□ 콘텐츠타입: 12=관광지, 14=문화시설, 15=행사/공연/축제
```

In [ ]:
# 🎭 대구 지역 행사/축제 정보
url_tour = "https://apis.data.go.kr/B551011/KorService1/searchFestival1"

params_tour = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "MobileOS": "ETC",
    "MobileApp": "TestApp",
    "_type": "json",
    "areaCode": "4",       # 대구
    "eventStartDate": datetime.now().strftime("%Y%m%d")  # 오늘 이후 행사
}

response_t = requests.get(url_tour, params=params_tour)

if check_response(response_t):
    data_t = response_t.json()
    items_t = data_t["response"]["body"]["items"]["item"]
    
    print("\n🎉 대구 행사/축제 목록:")
    for event in items_t:
        print(f"  🎭 {event['title']}")
        print(f"     기간: {event.get('eventstartdate','')} ~ {event.get('eventenddate','')}")
        print(f"     장소: {event.get('addr1','')}")

In [ ]:
# 🎯 미션: 지역 관광지 정보 조회
url_area = "https://apis.data.go.kr/B551011/KorService1/areaBasedList1"

params_area = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "MobileOS": "ETC",
    "MobileApp": "TestApp",
    "_type": "json",
    "areaCode": "4",        # 대구
    "contentTypeId": "12"   # 12=관광지, 14=문화시설, 15=축제행사
}

response_a = requests.get(url_area, params=params_area)

if check_response(response_a):
    data_a = response_a.json()
    spots = data_a["response"]["body"]["items"]["item"]
    
    print("\n🗺️ 대구 관광지 목록:")
    for spot in spots:
        print(f"  📍 {spot['title']} | {spot.get('addr1','')}")

---
# 🚗 실습 5. 주차장 정보

**포털 검색:** `주차장 정보`  
**대구 주차장:** `대구 주차장`으로 검색

### 📝 탐색 미션
```
□ API 이름: ___________________________
□ URL: ___________________________
□ 필수 파라미터: ___________________________
□ 응답 필드 중 유료/무료 구분 필드: ___________________________
```

In [ ]:
# 🚗 주차장 정보 조회
# ※ API 문서를 보고 직접 URL과 파라미터를 채워보세요!

url_park = ""  # ← API 문서에서 찾은 URL 입력

params_park = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    # ← 필수 파라미터 추가
}

if url_park:
    response_p = requests.get(url_park, params=params_park)
    print("응답 코드:", response_p.status_code)
    if response_p.status_code == 200:
        pretty(response_p.json())
else:
    print("⚠️ URL을 먼저 입력하세요!")

---
# 💊 실습 6. 의약품 정보

**포털 검색:** `의약품개요정보`  
**포털 URL:** https://www.data.go.kr/data/15075057/openapi.do

### 📝 탐색 미션
```
□ 제품명으로 검색하는 파라미터: itemName
□ 업체명으로 검색하는 파라미터: entpName
□ 주요 응답 필드: itemName, entpName, efcyQesitm(효능), useMethodQesitm(용법)
```

In [ ]:
# 💊 의약품 정보 조회
url_drug = "https://apis.data.go.kr/1471000/DrugPrdtPrmsnInfoService04/getDrugPrdtPrmsnDtlInq04"

params_drug = {
    "serviceKey": API_KEY,
    "numOfRows": "5",
    "pageNo": "1",
    "type": "json",
    "itemName": "타이레놀"  # ← 검색할 약품명
}

response_d = requests.get(url_drug, params=params_drug)

if check_response(response_d):
    data_d = response_d.json()
    items_d = data_d["body"]["items"]
    
    for drug in items_d:
        print(f"💊 {drug['itemName']}")
        print(f"   제조사: {drug.get('entpName','')}")
        print(f"   효능: {drug.get('efcyQesitm','')[:80]}...")  
        print()

In [ ]:
# 🎯 미션: 내가 알고 있는 약 검색해보기
my_drug = "게보린"  # ← 원하는 약 이름으로 바꿔보세요
params_drug["itemName"] = my_drug

response_d2 = requests.get(url_drug, params=params_drug)
if check_response(response_d2):
    data_d2 = response_d2.json()
    print(f"'{my_drug}' 검색 결과:")
    pretty(data_d2)

---
# 🏆 자유 실습 - 내가 찾은 API

data.go.kr에서 **내가 관심 있는 데이터**를 직접 찾아서 실습해보세요!

### 탐색 → 실습 체크리스트
```
□ 1. data.go.kr에서 관심 키워드 검색
□ 2. 오픈 API 목록에서 원하는 것 선택
□ 3. 활용 신청 클릭
□ 4. Swagger UI에서 Try it out → 200 성공 확인
□ 5. 아래 셀에 파이썬 코드로 구현
□ 6. 응답 데이터 구조 파악 후 원하는 값 추출
```

### 💡 아이디어
- 🎬 영화진흥위원회 박스오피스 순위
- 🚇 지하철 실시간 위치
- 📚 도서관 장서 검색
- ⚽ 한국 축구 경기 결과
- 🌊 해수욕장 정보
- 🏠 부동산 실거래가

In [ ]:
# 🏆 내가 찾은 API 실습

# API 이름: _______________________
# 포털 URL: _______________________
# 내가 뽑고 싶은 데이터: _______________________

my_url = ""       # ← 내 API URL
my_params = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    # ← 추가 파라미터
}

if my_url:
    my_response = requests.get(my_url, params=my_params)
    print("응답 코드:", my_response.status_code)
    if my_response.status_code == 200:
        pretty(my_response.json())
else:
    print("⚠️ URL을 먼저 입력하세요!")

In [ ]:
# 원하는 데이터 추출 코드 작성

# 여기에 작성하세요!


---
# 📊 심화: pandas로 데이터 정리하기

위 실습에서 가져온 데이터를 DataFrame으로 만들어 분석해봅시다!

In [ ]:
# 예시: 병원 데이터를 DataFrame으로 분석
# (위 실습 3에서 데이터를 가져왔다고 가정)

# df = pd.DataFrame(items)   # items를 위에서 가져온 데이터로 교체

# # 기본 정보
# print(df.shape)            # 행, 열 수
# print(df.columns.tolist()) # 컬럼 목록
# print(df.head())           # 처음 5행

# # 진료과목별 병원 수
# print(df["clCdNm"].value_counts())

# # 특정 구 병원만 필터
# df_filtered = df[df["addr"].str.contains("중구")]
# print(df_filtered[["yadmNm", "addr", "telno"]])

print("주석을 해제하고 실행해보세요!")

---
# 📝 오늘 배운 것 정리

| 실습 | API | 새로 배운 것 | 어려웠던 점 |
|------|-----|------------|------------|
| 1. 날씨 | 기상청 | | |
| 2. 음식점 | 지자체 | | |
| 3. 병원/약국 | 심평원 | | |
| 4. 관광/행사 | 관광공사 | | |
| 5. 주차장 | (직접 찾기) | | |
| 6. 의약품 | 식약처 | | |
| 자유 | (직접 찾기) | | |

---
### 💡 핵심 정리
```python
# 모든 공공데이터 API의 공통 패턴
import requests

API_KEY = "본인키".strip()           # 1. 키 (앞뒤 공백 제거 필수!)
url = "https://apis.data.go.kr/..." # 2. URL (API 문서에서 확인)
params = {                          # 3. 파라미터 (필수값 확인)
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "_type": "json"
}
res = requests.get(url, params=params)  # 4. 요청
data = res.json()                       # 5. JSON 변환
items = data["response"]["body"]["items"]["item"]  # 6. 데이터 추출
df = pd.DataFrame(items)                # 7. DataFrame 변환
```

### 다음 시간 🔜
- folium으로 지도에 시각화하기
- matplotlib/plotly로 차트 그리기  
- 여러 API 데이터 합쳐서 분석하기